# Gold Layer: Enriched Fact Sales (Comment-Driven DQ)

**Pattern**: Incremental Stream + Holiday Enrichment + Metadata DQ  
**Sources**: Silver CRM Sales Details, Silver Holidays API

This notebook joins sales data with public holiday information and enforces quality rules from the Catalog metadata.

In [0]:
# 1. Include the Generic DQ Engine
%run "../utils/utils_quality_control"

### Step 0: Define Metadata (The 'Invisible' Rules)
Run this after the first execution to 'Tag' your columns. 
Rules: order_number (NOT NULL), sales_amount (>= 0)

In [0]:
target_table = "workspace.gold.fact_sales"

if spark.catalog.tableExists(target_table):
    spark.sql(f"ALTER TABLE {target_table} CHANGE COLUMN order_number COMMENT 'DQ: NOT NULL'")
    spark.sql(f"ALTER TABLE {target_table} CHANGE COLUMN sales_amount COMMENT 'DQ: >= 0'")
    print("✓ Sales metadata (DQ Tags) updated in Catalog.")
else:
    print("ℹ Table does not exist yet. Run the stream first to create it.")

In [0]:
from pyspark.sql.functions import col, lit, to_date, when
from delta.tables import DeltaTable

# 2. Configuration
quarantine_table = "workspace.gold.quarantine_sales"
checkpoint_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/gold_fact_sales"
holiday_table = "workspace.silver.holidays_cdc"

def process_gold_sales(batch_df, batch_id):
    # A. Enrichment (Join with Holidays)
    # Note: We use 'order_date' which exists in the silver table
    holidays_df = spark.table(holiday_table)
    
    enriched_df = batch_df.alias("s").join(
        holidays_df.alias("h"),
        to_date(col("s.order_date")) == col("h.date"),
        "left"
    ).select(
        col("s.*"),
        when(col("h.holiday_name").isNotNull(), True).otherwise(False).alias("is_holiday"),
        col("h.holiday_name")
    )
    
    # B. Apply Metadata DQ
    table_exists = spark.catalog.tableExists(target_table)
    if table_exists:
        validated_df = apply_metadata_dq(enriched_df, target_table)
    else:
        print("First Run: Creating target table...")
        validated_df = enriched_df.withColumn("is_invalid", lit(False)).withColumn("dq_errors", lit(""))
    
    # C. Split: Clean vs Quarantine
    clean_df = validated_df.filter("is_invalid = false").drop("is_invalid", "dq_errors")
    quarantine_df = validated_df.filter("is_invalid = true")
    
    # D. Save Quarantine
    if quarantine_df.count() > 0:
        print(f"⚠ {quarantine_df.count()} records quarantined.")
        quarantine_df.write.format("delta").mode("append").saveAsTable(quarantine_table)
    
    # E. Save Clean data
    if not table_exists:
        clean_df.write.format("delta").saveAsTable(target_table)
    else:
        gold_table = DeltaTable.forName(spark, target_table)
        (gold_table.alias("t")
            .merge(clean_df.alias("s"), "t.order_number = s.order_number AND t.product_key = s.product_key")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

# 3. Execute Stream
print(f"Starting Enriched Gold processing for {target_table}...")
query = (
    spark.readStream
    .table("workspace.silver.crm_sales_details_cdc")
    .writeStream
    .foreachBatch(process_gold_sales)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print(f"✓ Gold processing complete. Total records in {target_table}: {spark.table(target_table).count():,}")